In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

In [ ]:
# 1. Data Preparation
data = load_wine()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
# 2. Binary SVM Class (The Engine)
class BinarySVM:
    def __init__(self, learning_rate=0.001, lambda_param=0.01, n_iters=1000):
        self.lr = learning_rate
        self.lambda_param = lambda_param
        self.n_iters = n_iters
        self.w = None
        self.b = None

    def fit(self, X, y):
        # Ensure labels are 1 and -1
        y_transformed = np.where(y <= 0, -1, 1)
        n_samples, n_features = X.shape
        
        # Initialize weights and bias
        self.w = np.zeros(n_features)
        self.b = 0

        # Gradient Descent
        for _ in range(self.n_iters):
            for idx, x_i in enumerate(X):
                # Condition: y_i * (w*x_i - b) >= 1
                condition = y_transformed[idx] * (np.dot(x_i, self.w) - self.b) >= 1
                
                if condition:
                    # Point is outside the margin - only apply regularization
                    self.w -= self.lr * (2 * self.lambda_param * self.w)
                else:
                    # Point is inside or wrong side - apply hinge loss gradient
                    self.w -= self.lr * (2 * self.lambda_param * self.w - np.dot(x_i, y_transformed[idx]))
                    self.b -= self.lr * y_transformed[idx]

    def get_score(self, X):
        # Returns the raw distance from the boundary (the score)
        return np.dot(X, self.w) - self.b

In [ ]:
# 3. Multiclass Wrapper (One-vs-Rest)
class MulticlassSVM:
    def __init__(self, n_classes):
        self.n_classes = n_classes
        self.clfs = []

    def fit(self, X, y):
        for i in range(self.n_classes):
            # Create a binary target: 1 for class 'i', -1 for everything else
            y_binary = np.where(y == i, 1, -1)
            
            clf = BinarySVM(learning_rate=0.001, lambda_param=0.01, n_iters=1000)
            clf.fit(X, y_binary)
            self.clfs.append(clf)

    def predict(self, X):
        # Get scores from all binary classifiers
        # Resulting shape: (n_samples, n_classes)
        all_scores = np.array([clf.get_score(X) for clf in self.clfs]).T
        
        # Final label is the index of the highest score
        return np.argmax(all_scores, axis=1)

In [ ]:
# 4. Execution
print("Training One-vs-Rest SVMs...")
model = MulticlassSVM(n_classes=3)
model.fit(X_train, y_train)

# 5. Evaluation
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

train_acc = accuracy_score(y_train, y_train_pred)
test_acc = accuracy_score(y_test, y_test_pred)

print("\n=== SVM Results ===")
print(f"Accuracy (Train): {train_acc * 100:.2f}%")
print(f"Accuracy (Test):  {test_acc * 100:.2f}%")

In [ ]:
Training One-vs-Rest SVMs...

=== SVM Results ===
Accuracy (Train): 99.30%
Accuracy (Test):  100.00%